# 30 — Brier Score Comparison

The key question: is our model more accurate than the market? Compare Brier scores with confidence intervals.

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import match_ticker_to_game
from nba_edge.backtest.engine import BacktestEngine
from nba_edge.backtest.metrics import brier_score, compute_brier_scores, calibration_bins
from nba_edge.models.analytical import AnalyticalWinProb

In [ ]:
# Run backtest (same as 21)
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)
engine = BacktestEngine(model=AnalyticalWinProb(variance_rate=0.44))
game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]

results = []
for ticker in sorted(game_tickers):
    parsed = parse_ticker(ticker)
    if not parsed.game_date:
        continue
    try:
        available = discover_game_ids(parsed.game_date)
    except Exception:
        continue
    matched = match_ticker_to_game(ticker, available)
    if not matched:
        continue
    snapshots = load_boxscores_for_game(matched, parsed.game_date)
    if len(snapshots) < 50 or snapshots[-1].get('game_status', 0) != 3:
        continue
    ticker_trades = df.filter(pl.col('market_ticker') == ticker)
    result = engine.run_game(ticker_trades, snapshots, ticker)
    if result and len(result.trades) > 0:
        results.append(result)

print(f'Backtested {len(results)} markets')

In [ ]:
# Aggregate Brier scores
brier_result = compute_brier_scores(results)
print('=== BRIER SCORE COMPARISON ===')
print(f"  Model:  {brier_result['model_brier']:.6f}")
print(f"  Market: {brier_result['market_brier']:.6f}")
print(f"  Diff:   {brier_result['model_brier'] - brier_result['market_brier']:.6f}")
print(f"  N:      {brier_result['n_observations']:,}")
print()
if brier_result['model_brier'] < brier_result['market_brier']:
    print('  *** MODEL WINS — we have a potential edge ***')
else:
    improvement_needed = brier_result['model_brier'] - brier_result['market_brier']
    print(f'  Market is better by {improvement_needed:.6f}')
    print('  Need a better model to find edge.')

In [ ]:
# Bootstrap confidence intervals on Brier score difference
model_probs = []
market_probs = []
outcomes = []
for r in results:
    for t in r.trades:
        model_probs.append(t.model_prob_yes)
        market_probs.append(t.market_implied)
        outcomes.append(float(r.outcome_yes))

model_probs = np.array(model_probs)
market_probs = np.array(market_probs)
outcomes = np.array(outcomes)

n_bootstrap = 1000
diffs = []
n = len(outcomes)
rng = np.random.default_rng(42)

for _ in range(n_bootstrap):
    idx = rng.integers(0, n, size=n)
    model_bs = np.mean((model_probs[idx] - outcomes[idx]) ** 2)
    market_bs = np.mean((market_probs[idx] - outcomes[idx]) ** 2)
    diffs.append(model_bs - market_bs)

diffs = np.array(diffs)
ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
print(f'Brier score difference (model - market):')
print(f'  Mean: {np.mean(diffs):.6f}')
print(f'  95% CI: [{ci_lo:.6f}, {ci_hi:.6f}]')
print(f'  (Negative = model better, Positive = market better)')
if ci_hi < 0:
    print('  *** Statistically significant: model is better ***')
elif ci_lo > 0:
    print('  Statistically significant: market is better')
else:
    print('  Not statistically significant — CI spans zero')

In [ ]:
# Per-game Brier comparison
print(f"{'Ticker':45s} {'Model':>8s} {'Market':>8s} {'Diff':>8s} {'Better':>8s}")
print('-' * 80)

for r in results:
    m_probs = [t.model_prob_yes for t in r.trades]
    mk_probs = [t.market_implied for t in r.trades]
    outs = [r.outcome_yes] * len(r.trades)
    m_brier = brier_score(m_probs, outs)
    mk_brier = brier_score(mk_probs, outs)
    better = 'Model' if m_brier < mk_brier else 'Market'
    print(f'{r.market_ticker:45s} {m_brier:>8.4f} {mk_brier:>8.4f} {m_brier-mk_brier:>+8.4f} {better:>8s}')

model_wins = sum(1 for r in results 
                 if brier_score([t.model_prob_yes for t in r.trades], [r.outcome_yes]*len(r.trades)) <
                    brier_score([t.market_implied for t in r.trades], [r.outcome_yes]*len(r.trades)))
print(f'\nModel wins {model_wins}/{len(results)} games')

In [ ]:
# Brier score by time remaining (is model better early or late game?)
time_cuts = [2400, 1800, 1200, 600, 300, 120, 60]

print(f"{'Secs Remaining':>15s} {'Model BS':>10s} {'Market BS':>10s} {'Diff':>10s} {'N':>8s}")
print('-' * 58)

prev_cut = 3000
for cut in time_cuts:
    mask = (np.array([t.seconds_remaining for r in results for t in r.trades]) >= cut) & \
           (np.array([t.seconds_remaining for r in results for t in r.trades]) < prev_cut)
    if mask.sum() > 0:
        m_bs = np.mean((model_probs[mask] - outcomes[mask]) ** 2)
        mk_bs = np.mean((market_probs[mask] - outcomes[mask]) ** 2)
        print(f'{cut:>15d} {m_bs:>10.4f} {mk_bs:>10.4f} {m_bs-mk_bs:>+10.4f} {mask.sum():>8d}')
    prev_cut = cut